In [5]:
import pandas as pd 
import os 
import re
import numpy as np 
import warnings
from pandas.errors import SettingWithCopyWarning
warnings.simplefilter(action="ignore", category=SettingWithCopyWarning)
warnings.simplefilter(action="ignore", category=FutureWarning)
pd.options.display.max_columns = 100
pd.options.display.max_rows = 60


# 0. Preface
This notebook will preprocess the eICU dataset into a features matrix that you can use for machine learning. 

# 1. Introduction: the eICU Dataset

The eICU Collaborative Research Database is a large, real-world ICU dataset containing detailed information about patient care across many hospitals in the United States. It includes vital signs, laboratory measurements, diagnoses, treatments, and severity scores, all collected during ICU stays.

Unlike a typical “single table” dataset, eICU is stored as a **relational database**. This means the data is split across multiple tables, each representing a different aspect of patient care. These tables are linked together using shared identifiers such as:

* `patientunitstayid` → a single ICU visit (also called a “unit”) within a hospitalization
* `patienthealthsystemstayid` → a hospitalization visit
* `hospitalid` → the hospital itself (there are many hospitals in this dataset)

Relational databases are common, especially in healthcare. The key idea is that for each patient’s ICU visit (`patientunitstayid`), their data is spread across many tables, often with multiple rows per table. For example, in `lab.csv.gz`, a patient may have many lab results recorded during a single ICU stay.

---

### Table outline

While there are many tables in the eICU dataset, we will focus on a few core ones:

* `patient.csv.gz` → patient demographics and hospital-level information  
* `diagnosis.csv.gz` → diagnoses (ICD codes) recorded during the ICU stay  
* `admissionDx.csv.gz` → admission diagnoses in unstructured text (right before ICU admission)  
* `lab.csv.gz` → lab test results (e.g., creatinine, WBC)  
* `vitalPeriodic.csv.gz` and `vitalAperiodic.csv.gz` → vital signs (e.g., heart rate, blood pressure) recorded over time  

You are strongly encouraged to explore the full table descriptions here:  
https://eicu-crd.mit.edu/eicutables/admissiondrug/

Helpful example notebooks are available here:  
https://github.com/MIT-LCP/eicu-code/tree/main/notebooks

---

### Creating the cohort

This dataset can be processed into features \(X\) in many different ways. In this project, we will use the following setup:

* We consider patients who were in the hospital for at least 48 hours  
* We ensure that features \(X\) occur before the outcome \(Y\) (so the model is actually predictive)  
* Features \(X\): diagnoses, vitals, and labs recorded during hours 0–24 of the hospital stay (the first day)  
* Outcomes \(Y\): events observed during hours 24–48 (the second day), e.g., patient mortality  

---

### A note on timing

Timing in this dataset can be a bit confusing.

For each hospitalization (`patienthealthsystemstayid`), a patient may have multiple ICU stays (`patientunitstayid`). In this project, we focus only on ICU data within the first 48 hours of the initial hospitalization.

To determine when events (labs, diagnoses, vitals, etc.) occur, we use the *offset* columns in each table, which record the number of minutes from a reference time. These offsets allow us to align events across tables and restrict them to the time windows defined above.

We only consider features \(X\) recorded at any ICU visit within the first 24 hours of hospitalization; for outcomes \(Y\) we can take any values observed during any ICU visit within 24-48 hours of hospitalization.

In [9]:
# Flag as true if you are working witgh the eICU demo dataset; False if you have the full dataset 
demo = False 

# Make sure you are in the directory where you downloaded the data
if demo:
    data_dir = '../data/eicu-collaborative-research-database-demo-2.0.1/' #TODO: make sure this points to your directory containing the demo dataset
else:
    data_dir = '../data/physionet.org/files/eicu-crd/2.0/' #TODO: make sure this points to your directory containing the full dataset

assert os.path.exists(data_dir), f"Data directory {data_dir} does not exist. Please check that you've downloaded the data to the correct path name." 
needed_files = ['diagnosis.csv.gz', 'vitalAperiodic.csv.gz', 
                'admissionDx.csv.gz', 'hospital.csv.gz', 'vitalPeriodic.csv.gz', 
                'patient.csv.gz', 'treatment.csv.gz', 'apacheApsVar.csv.gz', 
                'lab.csv.gz', 'apachePredVar.csv.gz', 
                'apachePatientResult.csv.gz']
assert np.setdiff1d(needed_files, os.listdir(data_dir)).shape[0] == 0 # Folder should contain all the needed files. If not, check that you've downloaded the data correctly.


# 2. Load & clean the data 
We will create a table / matrix / feature set `patient_agg` where each row is a data point and the columns are all possible predictive features X or labels Y we might care about. That way, when we train our ML models, we will have all our data easily accessible in one place! 

### Patient table

The `patient.csv.gz` table serves as the "anchor" table for the dataset. Each row corresponds to a single ICU stay (`patientunitstayid`) and includes basic information about the patient and their hospitalization.

This table contains:
* Demographics (e.g., age, gender)
* Admission and discharge information
* Hospital identifiers (e.g., `hospitalid`)
* Timing information (e.g., length of stay, admission offsets)

In practice, you will almost always start with the `patient` table when defining a cohort, and then join other tables (labs, vitals, diagnoses, etc.) using `patientunitstayid` to gather additional features.

In [10]:
patient = pd.read_csv(f'{data_dir}/patient.csv.gz')
patient

,patientunitstayid,patienthealthsystemstayid,gender,age,ethnicity,hospitalid,wardid,apacheadmissiondx,admissionheight,hospitaladmittime24,hospitaladmitoffset,hospitaladmitsource,hospitaldischargeyear,hospitaldischargetime24,hospitaldischargeoffset,hospitaldischargelocation,hospitaldischargestatus,unittype,unitadmittime24,unitadmitsource,unitvisitnumber,unitstaytype,admissionweight,dischargeweight,unitdischargetime24,unitdischargeoffset,unitdischargelocation,unitdischargestatus,uniquepid
0,141168,128919,Female,70,Caucasian,59,91,"Rhythm disturbance (atrial, supraventricular)",152.4,15:54:00,0,Direct Admit,2015,03:50:00,3596,Death,Expired,Med-Surg ICU,15:54:00,Direct Admit,1,admit,84.3,85.8,03:50:00,3596,Death,Expired,002-34851
1,141178,128927,Female,52,Caucasian,60,83,NaN,162.6,08:56:00,-14,Emergency Department,2015,19:20:00,2050,Home,Alive,Med-Surg ICU,09:10:00,Emergency Department,1,admit,54.4,54.4,09:18:00,8,Step-Down Unit (SDU),Alive,002-33870
2,141179,128927,Female,52,Caucasian,60,83,NaN,162.6,08:56:00,-22,Emergency Department,2015,19:20:00,2042,Home,Alive,Med-Surg ICU,09:18:00,ICU to SDU,2,stepdown/other,NaN,60.4,19:20:00,2042,Home,Alive,002-33870
3,141194,128941,Male,68,Caucasian,73,92,"Sepsis, renal/UTI (including bladder)",180.3,18:18:40,-780,Floor,2015,23:30:00,12492,Home,Alive,CTICU,07:18:00,Floor,1,admit,73.9,76.7,15:31:00,4813,Floor,Alive,002-5276
4,141196,128943,Male,71,Caucasian,67,109,NaN,162.6,20:21:00,-99,Emergency Department,2015,17:00:00,5460,Home,Alive,Med-Surg ICU,22:00:00,ICU to SDU,2,stepdown/other,NaN,63.2,22:23:00,1463,Floor,Alive,002-37665
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200854,3353235,2743084,Male,50,Caucasian,458,1109,"CHF, congestive heart failure",175.3,04:55:00,-34,Emergency Department,2014,21:00:00,3811,Home,Alive,Cardiac ICU,05:29:00,Emergency Department,1,admit,90.0,99.2,23:18:00,1069,Telemetry,Alive,035-16382
200855,3353237,2743086,Female,79,Caucasian,458,1106,"Embolus, pulmonary",162.6,01:45:00,-14,Direct Admit,2014,19:04:00,9665,Home,Alive,MICU,01:59:00,Direct Admit,1,admit,78.4,81.4,23:08:00,1269,Step-Down Unit (SDU),Alive,035-751
200856,3353251,2743099,Male,73,African American,458,1104,Cardiac arrest (with or without respiratory ar...,177.8,12:51:00,-206,Emergency Department,2014,22:35:00,19098,Home,Alive,Cardiac ICU,16:17:00,Emergency Department,1,admit,102.0,96.2,23:16:00,16259,Telemetry,Alive,035-5166
200857,3353254,2743102,Male,81,Caucasian,459,1108,"Bleeding, lower GI",185.4,07:43:00,-271,Emergency Department,2015,18:38:00,6144,Home,Alive,Med-Surg ICU,12:14:00,Emergency Department,1,admit,83.9,92.9,19:25:00,431,Step-Down Unit (SDU),Alive,035-19511


In [11]:
# Total number of unique patients (using uniquepid, which identifies a person across stays)
N = patient.uniquepid.nunique()

# Define time windows (in hours)
hours_mortality = 48   # outcome window: we will look at mortality within 48 hours
hours_for_data = 24    # feature window: we only use data from the first 24 hours

# -----------------------------
# Step 1: Filter out short hospital visits < 24 hours 
# -----------------------------
# hospitaldischargeoffset is in minutes (time from hospital admission to discharge)
# We require patients to stay at least 24 hours so that we have enough data to build features
patient = patient[patient.hospitaldischargeoffset >= (hours_for_data * 60)]

print(f"Removed {N - patient.uniquepid.nunique()} patients who were discharged before {hours_for_data} hours")

Removed 11089 patients who were discharged before 24 hours


In [12]:
# Update patient count after filtering
N = patient.uniquepid.nunique()

# -----------------------------
# Step 2: Ensure valid 48h window
# -----------------------------
# hospitaladmitoffset is typically negative (minutes BEFORE ICU admission)
# -hospitaladmitoffset converts it into "minutes since hospital admission"
# We want ICU stays that occur within the first 48 hours of hospitalization
invalid = -patient.hospitaladmitoffset > hours_mortality * 60

print(f"Removing {invalid.sum()} ICU visits that start after {hours_mortality} hours = "
      f"Removing {N - patient.uniquepid.nunique()} patients who have no remaining data (usually 0)")

# Keep only valid ICU stays
patient = patient[~invalid]

Removing 35234 ICU visits that start after 48 hours = Removing 0 patients who have no remaining data (usually 0)


In [13]:
# -----------------------------
# Step 3: Define one possible outcome Y (mortality wihtin 48 hours)
# -----------------------------
# unitdischargestatus == 'Expired' means the patient died during that ICU stay
patient[f'mortality_at{hours_mortality}h'] = (patient.unitdischargestatus == 'Expired').astype(int)

# A patient can have multiple ICU stays within one hospitalization
# We aggregate to the hospitalization level:
# if the patient died in ANY ICU stay → mark the whole hospitalization as mortality = 1
patient[f'mortality_at{hours_mortality}h'] = (
    patient.groupby('patienthealthsystemstayid')[f'mortality_at{hours_mortality}h']
    .transform('max')
)
patient 

,patientunitstayid,patienthealthsystemstayid,gender,age,ethnicity,hospitalid,wardid,apacheadmissiondx,admissionheight,hospitaladmittime24,hospitaladmitoffset,hospitaladmitsource,hospitaldischargeyear,hospitaldischargetime24,hospitaldischargeoffset,hospitaldischargelocation,hospitaldischargestatus,unittype,unitadmittime24,unitadmitsource,unitvisitnumber,unitstaytype,admissionweight,dischargeweight,unitdischargetime24,unitdischargeoffset,unitdischargelocation,unitdischargestatus,uniquepid,mortality_at48h
0,141168,128919,Female,70,Caucasian,59,91,"Rhythm disturbance (atrial, supraventricular)",152.4,15:54:00,0,Direct Admit,2015,03:50:00,3596,Death,Expired,Med-Surg ICU,15:54:00,Direct Admit,1,admit,84.3,85.8,03:50:00,3596,Death,Expired,002-34851,1
1,141178,128927,Female,52,Caucasian,60,83,NaN,162.6,08:56:00,-14,Emergency Department,2015,19:20:00,2050,Home,Alive,Med-Surg ICU,09:10:00,Emergency Department,1,admit,54.4,54.4,09:18:00,8,Step-Down Unit (SDU),Alive,002-33870,0
2,141179,128927,Female,52,Caucasian,60,83,NaN,162.6,08:56:00,-22,Emergency Department,2015,19:20:00,2042,Home,Alive,Med-Surg ICU,09:18:00,ICU to SDU,2,stepdown/other,NaN,60.4,19:20:00,2042,Home,Alive,002-33870,0
3,141194,128941,Male,68,Caucasian,73,92,"Sepsis, renal/UTI (including bladder)",180.3,18:18:40,-780,Floor,2015,23:30:00,12492,Home,Alive,CTICU,07:18:00,Floor,1,admit,73.9,76.7,15:31:00,4813,Floor,Alive,002-5276,0
4,141196,128943,Male,71,Caucasian,67,109,NaN,162.6,20:21:00,-99,Emergency Department,2015,17:00:00,5460,Home,Alive,Med-Surg ICU,22:00:00,ICU to SDU,2,stepdown/other,NaN,63.2,22:23:00,1463,Floor,Alive,002-37665,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200854,3353235,2743084,Male,50,Caucasian,458,1109,"CHF, congestive heart failure",175.3,04:55:00,-34,Emergency Department,2014,21:00:00,3811,Home,Alive,Cardiac ICU,05:29:00,Emergency Department,1,admit,90.0,99.2,23:18:00,1069,Telemetry,Alive,035-16382,0
200855,3353237,2743086,Female,79,Caucasian,458,1106,"Embolus, pulmonary",162.6,01:45:00,-14,Direct Admit,2014,19:04:00,9665,Home,Alive,MICU,01:59:00,Direct Admit,1,admit,78.4,81.4,23:08:00,1269,Step-Down Unit (SDU),Alive,035-751,0
200856,3353251,2743099,Male,73,African American,458,1104,Cardiac arrest (with or without respiratory ar...,177.8,12:51:00,-206,Emergency Department,2014,22:35:00,19098,Home,Alive,Cardiac ICU,16:17:00,Emergency Department,1,admit,102.0,96.2,23:16:00,16259,Telemetry,Alive,035-5166,0
200857,3353254,2743102,Male,81,Caucasian,459,1108,"Bleeding, lower GI",185.4,07:43:00,-271,Emergency Department,2015,18:38:00,6144,Home,Alive,Med-Surg ICU,12:14:00,Emergency Department,1,admit,83.9,92.9,19:25:00,431,Step-Down Unit (SDU),Alive,035-19511,0


In [14]:
# -----------------------------
# Step 4: Encode admission diagnosis
# -----------------------------
# apacheadmissiondx is a categorical variable (text labels of admission diagnosis)
# that are made as a reason for why the patient was admitted to the ICU
# Convert it into one-hot encoded features (one column per diagnosis)
admissiondx_ohe = (
    pd.get_dummies(patient.apacheadmissiondx, dummy_na=True)
    .astype(int)
    .add_prefix('admissiondx_')
)

# Inspect most common diagnoses (by prevalence)
print(admissiondx_ohe.mean().sort_values(ascending=False)[:40])

# Replace original column with one-hot encoded features
patient = pd.concat(
    [patient.drop(columns='apacheadmissiondx'), admissiondx_ohe],
    axis=1
)

# Final processed patient table
patient

admissiondx_nan                                                                                                                             0.073030
admissiondx_Sepsis, pulmonary                                                                                                               0.048682
admissiondx_Infarction, acute myocardial (MI)                                                                                               0.044038
admissiondx_CVA, cerebrovascular accident/stroke                                                                                            0.039170
admissiondx_CHF, congestive heart failure                                                                                                   0.034817
admissiondx_Sepsis, renal/UTI (including bladder)                                                                                           0.031874
admissiondx_Diabetic ketoacidosis                                                                         

,patientunitstayid,patienthealthsystemstayid,gender,age,ethnicity,hospitalid,wardid,admissionheight,hospitaladmittime24,hospitaladmitoffset,hospitaladmitsource,hospitaldischargeyear,hospitaldischargetime24,hospitaldischargeoffset,hospitaldischargelocation,hospitaldischargestatus,unittype,unitadmittime24,unitadmitsource,unitvisitnumber,unitstaytype,admissionweight,dischargeweight,unitdischargetime24,unitdischargeoffset,unitdischargelocation,unitdischargestatus,uniquepid,mortality_at48h,"admissiondx_ARDS-adult respiratory distress syndrome, non-cardiogenic pulmonary edema",admissiondx_Abdomen only trauma,admissiondx_Abdomen/extremity trauma,admissiondx_Abdomen/face trauma,admissiondx_Abdomen/multiple trauma,admissiondx_Abdomen/pelvis trauma,admissiondx_Abdomen/spinal trauma,admissiondx_Ablation or mapping of cardiac conduction pathway,"admissiondx_Abscess, neurologic","admissiondx_Abscess/infection-cranial, surgery for",admissiondx_Acid-base/electrolyte disturbance,admissiondx_Addisons disease,admissiondx_Adrenal neoplasm (including pheochromocytoma),admissiondx_Adrenalectomy,admissiondx_Alcohol withdrawal,admissiondx_Amputation (non-traumatic),admissiondx_Amyotrophic lateral sclerosis,admissiondx_Anaphylaxis,"admissiondx_Anastomosis, vascular",admissiondx_Anemia,"admissiondx_Aneurysm repair, ventricular",...,admissiondx_Smoke inhalation,admissiondx_Spinal cord only trauma,"admissiondx_Spinal cord surgery, other",admissiondx_Spinal/extremity trauma,admissiondx_Spinal/face trauma,admissiondx_Spinal/multiple trauma,admissiondx_Splenectomy,admissiondx_Stereotactic procedure,admissiondx_Subarachnoid hemorrhage/arteriovenous malformation,admissiondx_Subarachnoid hemorrhage/intracranial aneurysm,"admissiondx_Subarachnoid hemorrhage/intracranial aneurysm, surgery for","admissiondx_TURP, transurethral prostate resection for benign prostatic hypertrophy","admissiondx_TURP, transurethral prostate resection for cancer","admissiondx_Tamponade, pericardial","admissiondx_Thoracotomy for benign tumor (i.e. mediastinal chest wall mass, thymectomy)",admissiondx_Thoracotomy for bronchopleural fistula,admissiondx_Thoracotomy for esophageal cancer,admissiondx_Thoracotomy for lung cancer,admissiondx_Thoracotomy for lung reduction,admissiondx_Thoracotomy for other malignancy in chest,admissiondx_Thoracotomy for other reasons,admissiondx_Thoracotomy for pleural disease,admissiondx_Thoracotomy for thoracic/respiratory infection,admissiondx_Thrombectomy (with general anesthesia),admissiondx_Thrombectomy (without general anesthesia),admissiondx_Thrombocytopenia,"admissiondx_Thrombosis, vascular (deep vein)","admissiondx_Thrombus, arterial",admissiondx_Thyroid neoplasm,admissiondx_Thyroidectomy,admissiondx_Thyroidectomy and Parathyroidectomy,"admissiondx_Toxicity, drug (i.e., beta blockers, calcium channel blockers, etc.)",admissiondx_Tracheostomy,admissiondx_Transphenoidal surgery,"admissiondx_Transplant, other","admissiondx_Trauma medical, other","admissiondx_Trauma surgery, other",admissiondx_Tricuspid valve surgery,"admissiondx_Tumor removal, intracardiac","admissiondx_Ulcer disease, peptic","admissiondx_Vascular medical, other","admissiondx_Vascular surgery, other",admissiondx_Vasculitis,admissiondx_Vena cava clipping,admissiondx_Vena cava filter insertion,admissiondx_Ventricular Septal Defect (VSD) Repair,admissiondx_Ventriculostomy,admissiondx_Weaning from mechanical ventilation (transfer from other unit or hospital only),admissiondx_Whipple-surgery for pancreatic cancer,admissiondx_nan
0,141168,128919,Female,70,Caucasian,59,91,152.4,15:54:00,0,Direct Admit,2015,03:50:00,3596,Death,Expired,Med-Surg ICU,15:54:00,Direct Admit,1,admit,84.3,85.8,03:50:00,3596,Death,Expired,002-34851,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,141178,128927,Female,52,Caucasian,60,83,162.6,08:56:00,-14,Emergency Department,2015,19:20:00,2050,Home,Alive,Med-Surg ICU,0

In [15]:
patient.drop(columns=['wardid', 'dischargeweight', 'hospitaladmittime24', 'hospitaladmitsource',
                      'unittype', 'unitadmittime24', 'unitadmitsource','unitdischargetime24', 'hospitaldischargetime24', 
                      'unitdischargeoffset', 
                      'hospitaldischargeoffset', 'hospitaldischargelocation', 'unitstaytype', 'unitdischargelocation', ], inplace=True)
patient

,patientunitstayid,patienthealthsystemstayid,gender,age,ethnicity,hospitalid,admissionheight,hospitaladmitoffset,hospitaldischargeyear,hospitaldischargestatus,unitvisitnumber,admissionweight,unitdischargestatus,uniquepid,mortality_at48h,"admissiondx_ARDS-adult respiratory distress syndrome, non-cardiogenic pulmonary edema",admissiondx_Abdomen only trauma,admissiondx_Abdomen/extremity trauma,admissiondx_Abdomen/face trauma,admissiondx_Abdomen/multiple trauma,admissiondx_Abdomen/pelvis trauma,admissiondx_Abdomen/spinal trauma,admissiondx_Ablation or mapping of cardiac conduction pathway,"admissiondx_Abscess, neurologic","admissiondx_Abscess/infection-cranial, surgery for",admissiondx_Acid-base/electrolyte disturbance,admissiondx_Addisons disease,admissiondx_Adrenal neoplasm (including pheochromocytoma),admissiondx_Adrenalectomy,admissiondx_Alcohol withdrawal,admissiondx_Amputation (non-traumatic),admissiondx_Amyotrophic lateral sclerosis,admissiondx_Anaphylaxis,"admissiondx_Anastomosis, vascular",admissiondx_Anemia,"admissiondx_Aneurysm repair, ventricular","admissiondx_Aneurysm, abdominal aortic","admissiondx_Aneurysm, abdominal aortic; with dissection","admissiondx_Aneurysm, abdominal aortic; with rupture","admissiondx_Aneurysm, dissecting aortic","admissiondx_Aneurysm, thoracic aortic","admissiondx_Aneurysm, thoracic aortic; with dissection","admissiondx_Aneurysm, thoracic aortic; with rupture","admissiondx_Aneurysm/pseudoaneurysm, other","admissiondx_Aneurysms, repair of other (except ventricular)","admissiondx_Angina, stable (asymp or stable pattern of symptoms w/meds)","admissiondx_Angina, unstable (angina interferes w/quality of life or meds are tolerated poorly)",admissiondx_Aortic and Mitral valve replacement,admissiondx_Aortic valve replacement (isolated),"admissiondx_Apnea, sleep",...,admissiondx_Smoke inhalation,admissiondx_Spinal cord only trauma,"admissiondx_Spinal cord surgery, other",admissiondx_Spinal/extremity trauma,admissiondx_Spinal/face trauma,admissiondx_Spinal/multiple trauma,admissiondx_Splenectomy,admissiondx_Stereotactic procedure,admissiondx_Subarachnoid hemorrhage/arteriovenous malformation,admissiondx_Subarachnoid hemorrhage/intracranial aneurysm,"admissiondx_Subarachnoid hemorrhage/intracranial aneurysm, surgery for","admissiondx_TURP, transurethral prostate resection for benign prostatic hypertrophy","admissiondx_TURP, transurethral prostate resection for cancer","admissiondx_Tamponade, pericardial","admissiondx_Thoracotomy for benign tumor (i.e. mediastinal chest wall mass, thymectomy)",admissiondx_Thoracotomy for bronchopleural fistula,admissiondx_Thoracotomy for esophageal cancer,admissiondx_Thoracotomy for lung cancer,admissiondx_Thoracotomy for lung reduction,admissiondx_Thoracotomy for other malignancy in chest,admissiondx_Thoracotomy for other reasons,admissiondx_Thoracotomy for pleural disease,admissiondx_Thoracotomy for thoracic/respiratory infection,admissiondx_Thrombectomy (with general anesthesia),admissiondx_Thrombectomy (without general anesthesia),admissiondx_Thrombocytopenia,"admissiondx_Thrombosis, vascular (deep vein)","admissiondx_Thrombus, arterial",admissiondx_Thyroid neoplasm,admissiondx_Thyroidectomy,admissiondx_Thyroidectomy and Parathyroidectomy,"admissiondx_Toxicity, drug (i.e., beta blockers, calcium channel blockers, etc.)",admissiondx_Tracheostomy,admissiondx_Transphenoidal surgery,"admissiondx_Transplant, other","admissiondx_Trauma medical, other","admissiondx_Trauma surgery, other",admissiondx_Tricuspid valve surgery,"admissiondx_Tumor removal, intracardiac","admissiondx_Ulcer disease, peptic","admissiondx_Vascular medical, other","admissiondx_Vascular surgery, other",admissiondx_Vasculitis,admissiondx_Vena cava clipping,admissiondx_Vena cava filter insertion,admissiondx_Ventricular Septal Defect (VSD) Repair,admissiondx_Ventriculostomy,admissiondx_Weaning from mechanical ventilation (transfer from other unit or hospital only),admissiondx_Whipple-surgery for pancreat

### Hospital table

The `hospital.csv.gz` table contains information about each hospital in the dataset. Each row corresponds to a hospital (`hospitalid`) and may include attributes such as region, hospital size, teaching status, and other structural characteristics.

This table is useful because the eICU dataset spans **many different hospitals**, each of which may have different:
* patient populations  
* clinical practices  
* resource availability  
* coding and data collection patterns  

By joining the `hospital` table with the `patient` table (using `hospitalid`), we can incorporate hospital-level features into our analysis.

This is especially important when thinking about **generalization and dataset shift**. For instance, a model trained on data from one set of hospitals may not perform well on another. 

In [16]:
hospital = pd.read_csv(f'{data_dir}/hospital.csv.gz')
hospital

,hospitalid,numbedscategory,teachingstatus,region
0,56,<100,f,Midwest
1,58,100 - 249,f,Midwest
2,59,<100,f,Midwest
3,60,<100,f,Midwest
4,61,<100,f,Midwest
...,...,...,...,...
203,447,NaN,f,Midwest
204,449,>= 500,t,Midwest
205,452,NaN,f,Midwest
206,458,>= 500,f,South


In [17]:
# Joint with the hospital table to get hospital-level features for each ICU visit
patient = patient.merge(hospital.drop(columns='teachingstatus').rename(
    columns={c: f"hospital_{c}" for c in hospital.columns if c != 'hospitalid'}), 
               on='hospitalid', how='left')
patient

,patientunitstayid,patienthealthsystemstayid,gender,age,ethnicity,hospitalid,admissionheight,hospitaladmitoffset,hospitaldischargeyear,hospitaldischargestatus,unitvisitnumber,admissionweight,unitdischargestatus,uniquepid,mortality_at48h,"admissiondx_ARDS-adult respiratory distress syndrome, non-cardiogenic pulmonary edema",admissiondx_Abdomen only trauma,admissiondx_Abdomen/extremity trauma,admissiondx_Abdomen/face trauma,admissiondx_Abdomen/multiple trauma,admissiondx_Abdomen/pelvis trauma,admissiondx_Abdomen/spinal trauma,admissiondx_Ablation or mapping of cardiac conduction pathway,"admissiondx_Abscess, neurologic","admissiondx_Abscess/infection-cranial, surgery for",admissiondx_Acid-base/electrolyte disturbance,admissiondx_Addisons disease,admissiondx_Adrenal neoplasm (including pheochromocytoma),admissiondx_Adrenalectomy,admissiondx_Alcohol withdrawal,admissiondx_Amputation (non-traumatic),admissiondx_Amyotrophic lateral sclerosis,admissiondx_Anaphylaxis,"admissiondx_Anastomosis, vascular",admissiondx_Anemia,"admissiondx_Aneurysm repair, ventricular","admissiondx_Aneurysm, abdominal aortic","admissiondx_Aneurysm, abdominal aortic; with dissection","admissiondx_Aneurysm, abdominal aortic; with rupture","admissiondx_Aneurysm, dissecting aortic","admissiondx_Aneurysm, thoracic aortic","admissiondx_Aneurysm, thoracic aortic; with dissection","admissiondx_Aneurysm, thoracic aortic; with rupture","admissiondx_Aneurysm/pseudoaneurysm, other","admissiondx_Aneurysms, repair of other (except ventricular)","admissiondx_Angina, stable (asymp or stable pattern of symptoms w/meds)","admissiondx_Angina, unstable (angina interferes w/quality of life or meds are tolerated poorly)",admissiondx_Aortic and Mitral valve replacement,admissiondx_Aortic valve replacement (isolated),"admissiondx_Apnea, sleep",...,"admissiondx_Spinal cord surgery, other",admissiondx_Spinal/extremity trauma,admissiondx_Spinal/face trauma,admissiondx_Spinal/multiple trauma,admissiondx_Splenectomy,admissiondx_Stereotactic procedure,admissiondx_Subarachnoid hemorrhage/arteriovenous malformation,admissiondx_Subarachnoid hemorrhage/intracranial aneurysm,"admissiondx_Subarachnoid hemorrhage/intracranial aneurysm, surgery for","admissiondx_TURP, transurethral prostate resection for benign prostatic hypertrophy","admissiondx_TURP, transurethral prostate resection for cancer","admissiondx_Tamponade, pericardial","admissiondx_Thoracotomy for benign tumor (i.e. mediastinal chest wall mass, thymectomy)",admissiondx_Thoracotomy for bronchopleural fistula,admissiondx_Thoracotomy for esophageal cancer,admissiondx_Thoracotomy for lung cancer,admissiondx_Thoracotomy for lung reduction,admissiondx_Thoracotomy for other malignancy in chest,admissiondx_Thoracotomy for other reasons,admissiondx_Thoracotomy for pleural disease,admissiondx_Thoracotomy for thoracic/respiratory infection,admissiondx_Thrombectomy (with general anesthesia),admissiondx_Thrombectomy (without general anesthesia),admissiondx_Thrombocytopenia,"admissiondx_Thrombosis, vascular (deep vein)","admissiondx_Thrombus, arterial",admissiondx_Thyroid neoplasm,admissiondx_Thyroidectomy,admissiondx_Thyroidectomy and Parathyroidectomy,"admissiondx_Toxicity, drug (i.e., beta blockers, calcium channel blockers, etc.)",admissiondx_Tracheostomy,admissiondx_Transphenoidal surgery,"admissiondx_Transplant, other","admissiondx_Trauma medical, other","admissiondx_Trauma surgery, other",admissiondx_Tricuspid valve surgery,"admissiondx_Tumor removal, intracardiac","admissiondx_Ulcer disease, peptic","admissiondx_Vascular medical, other","admissiondx_Vascular surgery, other",admissiondx_Vasculitis,admissiondx_Vena cava clipping,admissiondx_Vena cava filter insertion,admissiondx_Ventricular Septal Defect (VSD) Repair,admissiondx_Ventriculostomy,admissiondx_Weaning from mechanical ventilation (transfer from other unit or hospital only),admissiondx_Whipple-surgery for pancreatic cancer,admissiondx_nan,hospital_numbedscategory,hospital_regio

In [18]:
### Aggregate multiple ICU stays into one row per hospitalization

# Recall:
# - patienthealthsystemstayid = one hospitalization
# - patientunitstayid         = one ICU stay within that hospitalization
#
# A single hospitalization can contain multiple ICU stays, including
# multiple ICU stays within our first 48-hour window.
#
# Since our prediction unit is the hospitalization (not the individual ICU stay),
# we want to collapse all ICU-stay-level rows into ONE row per
# patienthealthsystemstayid.

keep_cols = ['gender', 'age', 'ethnicity', 'hospitalid',
    'hospitaldischargeyear', 'mortality_at48h',
    'hospital_numbedscategory','hospital_region'
]

# These are the one-hot encoded admission diagnosis columns we created earlier.
admissiondx_cols = admissiondx_ohe.columns


patient_agg = patient.groupby('patienthealthsystemstayid').agg({
    **{c: 'first' for c in keep_cols},         # same within hospitalization so it does not matter whether we take the first, max, or min.
    **{c: 'max' for c in admissiondx_cols},    # keep diagnosis if it ever appears
    'admissionheight': 'mean',
    'admissionweight': 'mean',
}).reset_index()

# This prints the fraction of hospitalizations where mortality occurred
# within the 48-hour outcome window.
print(
    f"Mortality rate of patients that die between {hours_for_data}-{hours_mortality} hours: ",
    patient_agg[f'mortality_at{hours_mortality}h'].mean()
)

# Final hospitalization-level table
patient_agg

Mortality rate of patients that die between 24-48 hours:  0.03839355418403571


,patienthealthsystemstayid,gender,age,ethnicity,hospitalid,hospitaldischargeyear,mortality_at48h,hospital_numbedscategory,hospital_region,"admissiondx_ARDS-adult respiratory distress syndrome, non-cardiogenic pulmonary edema",admissiondx_Abdomen only trauma,admissiondx_Abdomen/extremity trauma,admissiondx_Abdomen/face trauma,admissiondx_Abdomen/multiple trauma,admissiondx_Abdomen/pelvis trauma,admissiondx_Abdomen/spinal trauma,admissiondx_Ablation or mapping of cardiac conduction pathway,"admissiondx_Abscess, neurologic","admissiondx_Abscess/infection-cranial, surgery for",admissiondx_Acid-base/electrolyte disturbance,admissiondx_Addisons disease,admissiondx_Adrenal neoplasm (including pheochromocytoma),admissiondx_Adrenalectomy,admissiondx_Alcohol withdrawal,admissiondx_Amputation (non-traumatic),admissiondx_Amyotrophic lateral sclerosis,admissiondx_Anaphylaxis,"admissiondx_Anastomosis, vascular",admissiondx_Anemia,"admissiondx_Aneurysm repair, ventricular","admissiondx_Aneurysm, abdominal aortic","admissiondx_Aneurysm, abdominal aortic; with dissection","admissiondx_Aneurysm, abdominal aortic; with rupture","admissiondx_Aneurysm, dissecting aortic","admissiondx_Aneurysm, thoracic aortic","admissiondx_Aneurysm, thoracic aortic; with dissection","admissiondx_Aneurysm, thoracic aortic; with rupture","admissiondx_Aneurysm/pseudoaneurysm, other","admissiondx_Aneurysms, repair of other (except ventricular)","admissiondx_Angina, stable (asymp or stable pattern of symptoms w/meds)","admissiondx_Angina, unstable (angina interferes w/quality of life or meds are tolerated poorly)",admissiondx_Aortic and Mitral valve replacement,admissiondx_Aortic valve replacement (isolated),"admissiondx_Apnea, sleep","admissiondx_Apnea-sleep; surgery for (i.e., UPPP - uvulopalatopharyngoplasty)",admissiondx_Appendectomy,"admissiondx_Arrest, respiratory (without cardiac arrest)","admissiondx_Arteriovenous malformation, surgery for","admissiondx_Arthritis, rheumatoid","admissiondx_Arthritis, septic",...,"admissiondx_Spinal cord surgery, other",admissiondx_Spinal/extremity trauma,admissiondx_Spinal/face trauma,admissiondx_Spinal/multiple trauma,admissiondx_Splenectomy,admissiondx_Stereotactic procedure,admissiondx_Subarachnoid hemorrhage/arteriovenous malformation,admissiondx_Subarachnoid hemorrhage/intracranial aneurysm,"admissiondx_Subarachnoid hemorrhage/intracranial aneurysm, surgery for","admissiondx_TURP, transurethral prostate resection for benign prostatic hypertrophy","admissiondx_TURP, transurethral prostate resection for cancer","admissiondx_Tamponade, pericardial","admissiondx_Thoracotomy for benign tumor (i.e. mediastinal chest wall mass, thymectomy)",admissiondx_Thoracotomy for bronchopleural fistula,admissiondx_Thoracotomy for esophageal cancer,admissiondx_Thoracotomy for lung cancer,admissiondx_Thoracotomy for lung reduction,admissiondx_Thoracotomy for other malignancy in chest,admissiondx_Thoracotomy for other reasons,admissiondx_Thoracotomy for pleural disease,admissiondx_Thoracotomy for thoracic/respiratory infection,admissiondx_Thrombectomy (with general anesthesia),admissiondx_Thrombectomy (without general anesthesia),admissiondx_Thrombocytopenia,"admissiondx_Thrombosis, vascular (deep vein)","admissiondx_Thrombus, arterial",admissiondx_Thyroid neoplasm,admissiondx_Thyroidectomy,admissiondx_Thyroidectomy and Parathyroidectomy,"admissiondx_Toxicity, drug (i.e., beta blockers, calcium channel blockers, etc.)",admissiondx_Tracheostomy,admissiondx_Transphenoidal surgery,"admissiondx_Transplant, other","admissiondx_Trauma medical, other","admissiondx_Trauma surgery, other",admissiondx_Tricuspid valve surgery,"admissiondx_Tumor removal, intracardiac","admissiondx_Ulcer disease, peptic","admissiondx_Vascular medical, other","admissiondx_Vascular surgery, other",admissiondx_Vasculitis,admissiondx_Vena cava clipping,admissiondx_Vena cava filter insertion,admissiondx_Ventricular Septal Defect (VSD) Repair,admissiondx_Ventriculostomy,admissiond

In [19]:
# This will be useful for grouping types of features X together 
feature_sets = {'admissiondx': admissiondx_cols} 

### Diagnosis table


The `diagnosis.csv.gz` table contains structured diagnosis codes (e.g., ICD codes) that are assigned to a patient during their ICU stay. ICD codes (International Classification of Diseases) - for example, `I21` → acute myocardial infarction (heart attack) - are standardized codes used to represent diagnoses and medical conditions, mostly used for billing purposes (which inject their own form of biases... but that's a problem for another day). You can learn more about specific what the codes mean here: https://www.icd10data.com/ICD10CM/Codes. 

In `diagnosis.csv.gz` each row represents a diagnosis associated with a specific ICU visit (`patientunitstayid`), and includes timing information via offset columns.

This table is particularly useful for both **covariates (X)** and **outcomes (Y)**, depending on how we use timing. Again, to be rigorous researchers, it should be that covariates generally occur before outcomes we are trying to predict.

- **Covariates (X):**  
  We use diagnoses (along with labs and vitals) recorded in the first 24 hours of the hospitalization. These represent the information available early in a patient’s stay.

- **Outcomes (Y):**  
  We can define outcomes using events that occur *after* this window. For example, we may care about whether a patient is later diagnosed with myocardial infarction during the second day. Since we haven't decided specifically which outcome we want to predict yet, we will take a subset of them!   

In [20]:
diagnosis = pd.read_csv(f'{data_dir}/diagnosis.csv.gz')

# Keep only diagnosis rows for ICU stays that are present in our patient cohort <48 hours
# This removes diagnosis records for ICU stays we are not analyzing
diagnosis = diagnosis[diagnosis.patientunitstayid.isin(patient.patientunitstayid.unique())]

# Join in hospitaladmitoffset from the patient table
# We need this to align diagnosis events to time since hospital admission
diagnosis = diagnosis.merge(
    patient[['patientunitstayid', 'hospitaladmitoffset']],
    on='patientunitstayid',
    how='left'
)

# Compute timing of each diagnosis relative to hospital admission
# diagnosisoffset = minutes from ICU admission to diagnosis event
# hospitaladmitoffset = minutes from hospital admission to ICU admission
#   (typically stored as a negative number)
# Therefore: hospitaladmitoffset + diagnosisoffset
# gives the number of minutes after hospital admission that the diagnosis occurred
diagnosis['time_from_admit'] = -diagnosis.hospitaladmitoffset + diagnosis.diagnosisoffset

In [21]:
# Extract ICD10 codes from the diagnosis code column
# ---------------------------------------------------------
# The column contains unstructured strings that generally look like {icd9 code}, {equivalent icd10 code}
# In pracrtice, it may contain one or more diagnosis codes
#
# We use regex to extract codes that look like ICD10 codes:
#   - one capital letter 
#   - followed by 2 digits
#   - optionally followed by a decimal and 1-4 alphanumeric characters
#
# Examples:
#   I21
#   E11.9
#   N39.0
pattern = r"\b[A-TV-Z][0-9]{2}(?:\.[0-9A-Z]{1,4})?\b"

diagnosis['icd9code_pp'] = diagnosis.icd9code.astype(str).apply(lambda x: re.findall(pattern, x))

# explode(): If a row contains multiple diagnosis codes, create one row per code
diagnosis = diagnosis.explode("icd9code_pp").rename(
    columns={"icd9code_pp": "icd10"}
).drop(columns=[
    'icd9code',
    'diagnosispriority',
    'activeupondischarge',
    'diagnosisstring',
    'diagnosisid'
])
diagnosis


,patientunitstayid,diagnosisoffset,hospitaladmitoffset,time_from_admit,icd10
0,141168,72,0,72,I25.10
1,141168,118,0,118,NaN
2,141168,72,0,72,J44.9
3,141168,118,0,118,J44.9
4,141168,118,0,118,I50.9
...,...,...,...,...,...
2037560,3353251,11304,-206,11510,N39.0
2037561,3353251,4080,-206,4286,A41.9
2037562,3353254,41,-271,312,N17.9
2037563,3353254,41,-271,312,K92.2


In [22]:
# Split diagnosis events into "before 24h" (covariates X) and "after 24h" (possible outcomes Y, if we wish)

diagnosis_before24h = diagnosis[diagnosis.time_from_admit <= hours_for_data * 60]

# Diagnoses observed after 24 hours can optionally be used as OUTCOMES (Y)
# For example, we might ask:
#   "Can we predict whether a patient will later receive an MI diagnosis
#    based on their first-day data?"
diagnosis_after24h = diagnosis[diagnosis.time_from_admit > hours_for_data * 60]

# Inspect early diagnosis events
diagnosis_before24h

,patientunitstayid,diagnosisoffset,hospitaladmitoffset,time_from_admit,icd10
0,141168,72,0,72,I25.10
1,141168,118,0,118,NaN
2,141168,72,0,72,J44.9
3,141168,118,0,118,J44.9
4,141168,118,0,118,I50.9
...,...,...,...,...,...
2037542,3353251,1201,-206,1407,N17.9
2037548,3353251,1201,-206,1407,I21.3
2037562,3353254,41,-271,312,N17.9
2037563,3353254,41,-271,312,K92.2


In [23]:
# First we will handle covariates X 

# One-hot encode / multiple-hot encode diagnoses observed in the first 24 hours
diagnosis_ohe_before24h = pd.concat(
    [
        diagnosis_before24h[['patientunitstayid']],
        pd.get_dummies(diagnosis_before24h.icd10)
    ],
    axis=1
).astype(int).groupby('patientunitstayid').max().reset_index()

# Inspect ICU-stay-level diagnosis features from the first 24h
diagnosis_ohe_before24h

,patientunitstayid,A02.0,A03.9,A04.4,A04.7,A05.9,A09,A15.0,A31.9,A35,A41.9,A48.1,A48.3,A54.42,A81.00,A86,A87.9,A92.31,B00.2,B00.4,B02.9,B15.9,B17.1,B17.9,B19.10,B20,B25.1,B27.09,B37.0,B37.2,B38.0,B39.2,B44.1,B45.0,B45.1,B49,B54,B58.2,B58.89,B59,B69.0,B95.0,B95.4,B95.5,B95.6,B95.7,B96.89,B97.4,C00.2,C06.9,...,S43.00,S43.31,S45.80,S53.00,S73.00,S83.00,S85.00,S93.0,T18.10,T38.3,T39.09,T39.1X,T40.5,T40.60,T42.3X,T42.4X,T43.01,T44.7X,T46.0,T46.0X,T46.1X,T48.6,T50.90,T51.0,T58.9,T60.0,T71.2,T80.5,T80.9,T81.3,T81.7,T85.7,T86.00,T86.10,T86.11,T86.20,T86.21,T86.40,T86.819,T86.899,T86.90,T88,V08,V42.7,V42.83,V62.84,Y38.6,Z21,Z94.4,Z94.83
0,141168,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,141203,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,141227,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,141229,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,141266,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121420,3353235,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
121421,3353237,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
121422,3353251,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
121423,3353254,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [24]:
# ---------------------------------------------------------
# Aggregate first-24h diagnosis features to the hospitalization level
# ---------------------------------------------------------
# diagnosis_ohe_before24h is currently one row per ICU stay
# But our prediction unit is one row per hospitalization
#
# So we:
# 1. Merge the ICU-stay diagnosis features onto the patient table
#    to recover patienthealthsystemstayid
# 2. Group by patienthealthsystemstayid
# 3. Take max() across ICU stays
#    -> if the diagnosis appears in any ICU stay within that hospitalization,
#       mark it as present
icd10_cols = diagnosis_ohe_before24h.columns.difference(['patientunitstayid'])

# Save the names of these diagnosis feature columns so we can reuse them later
feature_sets['icd10_before24h'] = icd10_cols

diagnosis_ohe_before24h_agg = patient.merge(
    diagnosis_ohe_before24h,
    on='patientunitstayid',
    how='left'
).groupby('patienthealthsystemstayid')[icd10_cols].max()

# View the most common ICD-10 diagnosis features in the first 24 hours
diagnosis_ohe_before24h_agg.mean().sort_values(ascending=False)

I10       0.134712
J96.00    0.123917
J18.9     0.090915
N17.9     0.088060
I50.9     0.070981
            ...   
E05.21    0.000008
D73.3     0.000008
D69.2     0.000008
D47.9     0.000008
G02       0.000008
Length: 776, dtype: float64

In [25]:
patient_agg = patient_agg.merge(diagnosis_ohe_before24h_agg, on= 'patienthealthsystemstayid', how='left')

In [26]:
# Next we will do the same for possible outcomes Y (diagnoses observed after 24h) 

# One-hot encode diagnoses observed after 24 hours
# These are not used as features here, but they may be useful as future labels
# or outcomes for prediction tasks
diagnosis_ohe_after24h = pd.concat(
    [
        diagnosis_after24h[['patientunitstayid']],
        pd.get_dummies(diagnosis_after24h.icd10)
    ],
    axis=1
).astype(int).groupby('patientunitstayid').max().add_prefix('label_').reset_index()

# Inspect ICU-stay-level diagnosis outcomes after 24h
diagnosis_ohe_after24h


,patientunitstayid,label_A02.0,label_A03.9,label_A04.4,label_A04.5,label_A04.7,label_A05.9,label_A09,label_A15.0,label_A18.84,label_A31.9,label_A32.11,label_A39.0,label_A41.9,label_A48.1,label_A48.3,label_A86,label_A87.9,label_A92.31,label_B00.2,label_B00.4,label_B02.9,label_B15.9,label_B17.1,label_B17.9,label_B19.10,label_B20,label_B25.0,label_B25.8,label_B27.09,label_B37.0,label_B37.2,label_B38.0,label_B39.2,label_B44.1,label_B45.0,label_B45.1,label_B49,label_B54,label_B58.2,label_B58.89,label_B59,label_B69.0,label_B95.0,label_B95.4,label_B95.5,label_B95.6,label_B95.7,label_B96.89,label_B97.4,...,label_S73.00,label_S83.00,label_S85.00,label_S92.25,label_S93.0,label_T18.10,label_T38.3,label_T39.09,label_T39.1X,label_T40.5,label_T40.60,label_T42.3X,label_T42.4X,label_T43.01,label_T44.7X,label_T46.0,label_T46.0X,label_T46.1X,label_T50.90,label_T51.0,label_T58.9,label_T60.0,label_T71.2,label_T80.5,label_T80.9,label_T81.3,label_T81.5,label_T81.6,label_T81.7,label_T85.7,label_T86.00,label_T86.10,label_T86.11,label_T86.20,label_T86.21,label_T86.40,label_T86.41,label_T86.810,label_T86.819,label_T86.890,label_T86.899,label_T86.90,label_T88,label_V08,label_V42.7,label_V42.83,label_V62.84,label_Z21,label_Z94.4,label_Z94.83
0,141203,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,141296,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,141340,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,141362,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,141515,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53205,3353190,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
53206,3353194,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
53207,3353213,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
53208,3353226,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [27]:
# We can use this later for outcome labels Y of our choice 
icd10_cols = diagnosis_ohe_after24h.columns.difference(['patientunitstayid'])
feature_sets['icd10_after24h'] = icd10_cols
diagnosis_ohe_after24h_agg = patient.merge(diagnosis_ohe_after24h, on='patientunitstayid', how='left'
              ).groupby('patienthealthsystemstayid')[icd10_cols].max()
diagnosis_ohe_after24h_agg.mean().sort_values(ascending=False)

label_J96.00    0.240840
label_I10       0.181944
label_N17.9     0.168776
label_J18.9     0.137677
label_I48.0     0.125572
                  ...   
label_G02       0.000019
label_G08       0.000019
label_G44.00    0.000019
label_G61.89    0.000019
label_A02.0     0.000019
Length: 779, dtype: float64

In [28]:
patient_agg = patient_agg.merge(diagnosis_ohe_after24h_agg, on= 'patienthealthsystemstayid', how='left')

### Vitals table 

The eICU dataset contains two types of vital sign tables:

* `vitalPeriodic.csv.gz` → vitals recorded at regular intervals (e.g., heart rate, blood pressure from monitors)  
* `vitalAperiodic.csv.gz` → vitals recorded irregularly (e.g., nurse-charted measurements)  

Both tables capture time-series data about a patient’s physiological state. In this project, we summarize these time-series into features by aggregating values over time windows (e.g., first 24 hours).


In [29]:
# Load both vital sign tables
vital_periodic = pd.read_csv(f'{data_dir}/vitalperiodic.csv.gz').drop(columns='vitalperiodicid')
vital_aperiodic = pd.read_csv(f'{data_dir}/vitalaperiodic.csv.gz').drop(columns='vitalaperiodicid')

In [30]:
# We will process both tables in the same way and store results here
vitals_dfs = []

# Define time bins (in hours → converted to minutes)
# We split the first 24 hours into two bins: [0–12), [12–24). You can choose to change these. 
hour_bins = [0, 12, 24]
time_bins = [h * 60 for h in hour_bins]  # convert hours → minutes
bin_labels = [f"bin{i+1}" for i in range(len(time_bins) - 1)]

# ---------------------------------------------------------
# Process each vitals table
# ---------------------------------------------------------
for df in [vital_periodic, vital_aperiodic]:

    # Keep only ICU stays in our cohort
    df = df[df.patientunitstayid.isin(patient.patientunitstayid.unique())]

    # Join patient info to compute time since hospital admission
    df = df.merge(
        patient[['patientunitstayid', 'patienthealthsystemstayid', 'hospitaladmitoffset']],
        on='patientunitstayid',
        how='left'
    )

    # Compute minutes since hospital admission
    df['time_from_admit'] = -df.hospitaladmitoffset + df.observationoffset

    # Keep only observations in the first 24 hours (feature window)
    df = df[df.time_from_admit <= hours_for_data * 60]

    # Assign each observation to a time bin
    df["time_bin"] = pd.cut(
        df["observationoffset"],
        bins=time_bins,
        labels=bin_labels,
        right=False,
        include_lowest=True
    )

    df = df.drop(columns=[
        'observationoffset',
        'hospitaladmitoffset',
        'time_from_admit',
        'patientunitstayid'
    ])

    # Aggregate vitals within each time bin
    # For each hospitalization and time bin:
    #   compute the mean of each vital sign
    df_expanded = df.groupby(
        ["patienthealthsystemstayid", "time_bin"]
    )[df.columns.difference(['patienthealthsystemstayid', 'time_bin'])] \
     .mean() \
     .unstack("time_bin")

    vitals_dfs.append(df_expanded)

# Split results for periodic and aperiodic vitals
vital_periodic_pp, vital_aperiodic_pp = vitals_dfs

# Check resulting feature shapes
vital_periodic_pp.shape, vital_aperiodic_pp.shape

((125280, 32), (122630, 20))

In [31]:
# Combine both periodic and aperiodic vitals into one table
vital = vital_periodic_pp.reset_index().merge(vital_aperiodic_pp.reset_index(), on='patienthealthsystemstayid', how='outer')
vital.columns = np.array(['patienthealthsystemstayid'] + [f"{col}_{bin}" for col, bin in vital.columns[1:]])
feature_sets['vitals'] = vital.columns[1:]

vital

/var/folders/gv/cvbvg4412fg914zd3_cc80lh0000gn/T/ipykernel_65022/1619381816.py:2: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  vital = vital_periodic_pp.reset_index().merge(vital_aperiodic_pp.reset_index(), on='patienthealthsystemstayid', how='outer')


,patienthealthsystemstayid,cvp_bin1,cvp_bin2,etco2_bin1,etco2_bin2,heartrate_bin1,heartrate_bin2,icp_bin1,icp_bin2,padiastolic_bin1,padiastolic_bin2,pamean_bin1,pamean_bin2,pasystolic_bin1,pasystolic_bin2,respiration_bin1,respiration_bin2,sao2_bin1,sao2_bin2,st1_bin1,st1_bin2,st2_bin1,st2_bin2,st3_bin1,st3_bin2,systemicdiastolic_bin1,systemicdiastolic_bin2,systemicmean_bin1,systemicmean_bin2,systemicsystolic_bin1,systemicsystolic_bin2,temperature_bin1,temperature_bin2,cardiacinput_bin1,cardiacinput_bin2,cardiacoutput_bin1,cardiacoutput_bin2,noninvasivediastolic_bin1,noninvasivediastolic_bin2,noninvasivemean_bin1,noninvasivemean_bin2,noninvasivesystolic_bin1,noninvasivesystolic_bin2,paop_bin1,paop_bin2,pvr_bin1,pvr_bin2,pvri_bin1,pvri_bin2,svr_bin1,svr_bin2,svri_bin1,svri_bin2
0,128919,NaN,NaN,NaN,NaN,132.776860,118.562500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,77.666667,90.454545,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,65.000000,NaN,76.000000,27.00,108.500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,128927,NaN,NaN,NaN,NaN,94.763889,89.992857,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,98.733333,96.000000,-0.008333,-0.000714,-0.353819,0.02000,-0.325347,0.000714,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,66.153846,83.5,79.769231,100.00,106.923077,140.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,128941,NaN,NaN,NaN,NaN,92.618321,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,26.824427,NaN,98.763359,NaN,NaN,NaN,NaN,NaN,NaN,NaN,34.0,NaN,46.0,NaN,84.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,45.379310,NaN,57.517241,NaN,85.896552,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,128943,NaN,NaN,NaN,NaN,82.141026,84.245763,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21.243590,27.508621,95.453333,95.635593,-0.308974,-0.231308,-0.844231,-1.14322,-0.508974,-0.850935,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,73.333333,70.0,92.333333,96.25,123.666667,137.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,128948,NaN,NaN,NaN,NaN,108.300000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,89.388889,NaN,-1.470000,NaN,-0.920000,NaN,0.430000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,60.000000,NaN,76.045455,NaN,98.590909,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125330,2625167,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,81.000000,NaN,108.000000,NaN,146.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
125331,2726664,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,66.000000,NaN,84.000000,NaN,117.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
125332,2730287,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,72.000000,NaN,87.000000,NaN,110.500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
125333,2732777,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,68.000000,NaN,96.333333,NaN,132.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [32]:
patient_agg = patient_agg.merge(vital, on='patienthealthsystemstayid', how='left')
patient_agg

,patienthealthsystemstayid,gender,age,ethnicity,hospitalid,hospitaldischargeyear,mortality_at48h,hospital_numbedscategory,hospital_region,"admissiondx_ARDS-adult respiratory distress syndrome, non-cardiogenic pulmonary edema",admissiondx_Abdomen only trauma,admissiondx_Abdomen/extremity trauma,admissiondx_Abdomen/face trauma,admissiondx_Abdomen/multiple trauma,admissiondx_Abdomen/pelvis trauma,admissiondx_Abdomen/spinal trauma,admissiondx_Ablation or mapping of cardiac conduction pathway,"admissiondx_Abscess, neurologic","admissiondx_Abscess/infection-cranial, surgery for",admissiondx_Acid-base/electrolyte disturbance,admissiondx_Addisons disease,admissiondx_Adrenal neoplasm (including pheochromocytoma),admissiondx_Adrenalectomy,admissiondx_Alcohol withdrawal,admissiondx_Amputation (non-traumatic),admissiondx_Amyotrophic lateral sclerosis,admissiondx_Anaphylaxis,"admissiondx_Anastomosis, vascular",admissiondx_Anemia,"admissiondx_Aneurysm repair, ventricular","admissiondx_Aneurysm, abdominal aortic","admissiondx_Aneurysm, abdominal aortic; with dissection","admissiondx_Aneurysm, abdominal aortic; with rupture","admissiondx_Aneurysm, dissecting aortic","admissiondx_Aneurysm, thoracic aortic","admissiondx_Aneurysm, thoracic aortic; with dissection","admissiondx_Aneurysm, thoracic aortic; with rupture","admissiondx_Aneurysm/pseudoaneurysm, other","admissiondx_Aneurysms, repair of other (except ventricular)","admissiondx_Angina, stable (asymp or stable pattern of symptoms w/meds)","admissiondx_Angina, unstable (angina interferes w/quality of life or meds are tolerated poorly)",admissiondx_Aortic and Mitral valve replacement,admissiondx_Aortic valve replacement (isolated),"admissiondx_Apnea, sleep","admissiondx_Apnea-sleep; surgery for (i.e., UPPP - uvulopalatopharyngoplasty)",admissiondx_Appendectomy,"admissiondx_Arrest, respiratory (without cardiac arrest)","admissiondx_Arteriovenous malformation, surgery for","admissiondx_Arthritis, rheumatoid","admissiondx_Arthritis, septic",...,etco2_bin1,etco2_bin2,heartrate_bin1,heartrate_bin2,icp_bin1,icp_bin2,padiastolic_bin1,padiastolic_bin2,pamean_bin1,pamean_bin2,pasystolic_bin1,pasystolic_bin2,respiration_bin1,respiration_bin2,sao2_bin1,sao2_bin2,st1_bin1,st1_bin2,st2_bin1,st2_bin2,st3_bin1,st3_bin2,systemicdiastolic_bin1,systemicdiastolic_bin2,systemicmean_bin1,systemicmean_bin2,systemicsystolic_bin1,systemicsystolic_bin2,temperature_bin1,temperature_bin2,cardiacinput_bin1,cardiacinput_bin2,cardiacoutput_bin1,cardiacoutput_bin2,noninvasivediastolic_bin1,noninvasivediastolic_bin2,noninvasivemean_bin1,noninvasivemean_bin2,noninvasivesystolic_bin1,noninvasivesystolic_bin2,paop_bin1,paop_bin2,pvr_bin1,pvr_bin2,pvri_bin1,pvri_bin2,svr_bin1,svr_bin2,svri_bin1,svri_bin2
0,128919,Female,70,Caucasian,59,2015,1,<100,Midwest,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,NaN,NaN,132.776860,118.562500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,77.666667,90.454545,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,65.000000,NaN,76.000000,27.000000,108.500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,128927,Female,52,Caucasian,60,2015,0,<100,Midwest,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,NaN,NaN,94.763889,89.992857,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,98.733333,96.000000,-0.008333,-0.000714,-0.353819,0.02000,-0.325347,0.000714,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,66.153846,83.500000,79.769231,100.000000,106.923077,140.500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,128941,Male,68,Caucasian,73,2015,0,>= 500,Midwest,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,NaN,NaN,92.618321,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,26.824427,NaN,98.763359,NaN,NaN,NaN,NaN,NaN,NaN,NaN,34.000000,NaN,46.00,NaN,84.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,45.379310,NaN,57.517241,NaN,85.896552,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,128943,Male,71,Caucasian,67,2015,

### Lab table
The `lab.csv.gz` table contains laboratory test results collected during a patient’s ICU stay. Each row corresponds to a specific lab measurement (e.g., creatinine, white blood cell count) taken at a particular time.

In [33]:
lab = pd.read_csv(f'{data_dir}/lab.csv.gz')

# Keep only lab measurements for ICU stays in our patient cohort
lab = lab[lab.patientunitstayid.isin(patient.patientunitstayid.unique())]

# Clean lab names slightly
lab["labname"] = lab["labname"].str.replace("-", '')

# Align lab measurements to hospital admission time
lab = lab.merge(
    patient[['patientunitstayid', 'hospitaladmitoffset']],
    on='patientunitstayid',
    how='left'
)
# Compute minutes since hospital admission
# labresultoffset = minutes from ICU admission
# hospitaladmitoffset = minutes from hospital admission to ICU admission (typically negative)
lab['time_from_admit'] = -lab.hospitaladmitoffset + lab.labresultoffset

# Keep only lab measurements from the first 24 hours (feature window)
lab = lab[lab.time_from_admit <= hours_for_data * 60]

# Inspect filtered lab data
lab

,labid,patientunitstayid,labresultoffset,labtypeid,labname,labresult,labresulttext,labmeasurenamesystem,labmeasurenameinterface,labresultrevisedoffset,hospitaladmitoffset,time_from_admit
1,50363251,141168,1133,3,PT INR,2.5,2.5,ratio,NaN,1208,0,1133
3,50363250,141168,1133,3,PT,26.6,26.6,sec,sec,1208,0,1133
5,56072056,141168,231,3,PT INR,1.7,1.7,ratio,NaN,268,0,231
8,47753731,141168,516,1,BUN,26.0,26,mg/dL,mg/dL,540,0,516
11,56072055,141168,231,3,PT,17.1,17.1,sec,sec,268,0,231
...,...,...,...,...,...,...,...,...,...,...,...,...
28417385,826336611,3353263,-37,3,PTT,25.0,25,sec,sec,54,-108,71
28417387,824772677,3353263,-7,3,MCH,27.0,27,pg,pg,6,-108,101
28417389,824772676,3353263,-7,3,RDW,13.3,13.3,%,%,6,-108,101
28417391,824772675,3353263,-7,3,WBC x 1000,6.4,6.4,K/mcL,K/uL,6,-108,101


In [34]:
lab.groupby(lab.labname).labmeasurenameinterface.value_counts()[:40] # ideally, these units should all be the same 

labname                   labmeasurenameinterface
24 h urine protein        mg/dL                       1281
                          mg/24 hrs                      5
                          mg/day                         5
                          mg/24 Hr                       4
                          mg/24HR                        3
                          mg/24Hr                        3
                          g/24Hr                         2
                          mg/24 hr                       2
                          g/24 hours                     1
                          mg/24h                         1
                          mg/24 h                        1
                          MG/24HR                        1
                          MG/24 HR                       1
24 h urine urea nitrogen  mg/dL                        117
                          g/24 hr                        1
                          g/24h                          1
ALT (S

In [35]:
# Aggregate lab measurements into features
# For each (patientunitstayid, labname), compute the average lab value
lab_expanded = (
    lab.groupby(["patientunitstayid", "labname"])["labresult"]
        .mean()
        .unstack()   # convert lab names into columns
        .reset_index()
)
# Note: Missing values (NaN) mean that lab was not measured for that patient
lab_expanded

labname,patientunitstayid,24 h urine protein,24 h urine urea nitrogen,ALT (SGPT),ANF/ANA,AST (SGOT),Acetaminophen,Amikacin peak,Amikacin random,Amikacin trough,BNP,BUN,Base Deficit,Base Excess,CPK,CPKMB,CPKMB INDEX,CRP,CRPhs,Carbamazepine,Carboxyhemoglobin,Clostridium difficile toxin A+B,Cyclosporin,Device,Digoxin,ESR,Fe,Fe/TIBC Ratio,Ferritin,FiO2,Gentamicin peak,Gentamicin random,Gentamicin trough,HCO3,HDL,HIV 1&2 AB,HSV 1&2 IgG AB,HSV 1&2 IgG AB titer,Hct,Hgb,LDH,LDL,LPM O2,Legionella pneumophila Ab,Lidocaine,Lithium,MCH,MCHC,MCV,MPV,...,calcium,cd 4,chloride,cortisol,creatinine,direct bilirubin,eos,ethanol,fibrinogen,folate,free T4,glucose,glucose CSF,haptoglobin,ionized calcium,lactate,lipase,lymphs,magnesium,monos,myoglobin,pH,paCO2,paO2,phosphate,platelets x 1000,polys,potassium,prealbumin,prolactin,protein CSF,protein C,protein S,reticulocyte count,salicylate,serum ketones,serum osmolality,sodium,total bilirubin,total cholesterol,total protein,transferrin,triglycerides,troponin I,troponin T,uric acid,urinary creatinine,urinary osmolality,urinary sodium,urinary specific gravity
0,141168,NaN,NaN,199.0,NaN,468.5,NaN,NaN,NaN,NaN,NaN,26.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.25,13.150,NaN,NaN,NaN,NaN,NaN,NaN,29.20,32.65,89.35,NaN,...,9.0,NaN,101.5,NaN,2.125,NaN,0.500000,NaN,NaN,NaN,NaN,113.0,NaN,NaN,NaN,NaN,NaN,12.500000,NaN,16.500000,NaN,NaN,NaN,NaN,NaN,211.0,70.5,4.100000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,139.0,3.35,NaN,7.10,NaN,NaN,NaN,NaN,8.1,NaN,NaN,NaN,NaN
1,141178,NaN,NaN,52.0,NaN,40.0,NaN,NaN,NaN,NaN,NaN,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,43.60,15.500,NaN,NaN,NaN,NaN,NaN,NaN,33.70,35.60,94.80,NaN,...,8.0,NaN,108.0,NaN,0.700,NaN,3.000000,234.0,NaN,NaN,NaN,77.0,NaN,NaN,NaN,NaN,NaN,45.000000,NaN,7.000000,NaN,NaN,NaN,NaN,NaN,273.0,45.0,3.600000,NaN,NaN,NaN,NaN,NaN,NaN,2.3,NaN,NaN,146.0,0.40,NaN,7.40,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,141179,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,8.2,NaN,107.5,NaN,0.700,NaN,NaN,NaN,NaN,NaN,NaN,72.0,NaN,NaN,NaN,NaN,NaN,NaN,1.900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.900000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,143.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,141194,NaN,NaN,19.5,NaN,19.5,NaN,NaN,NaN,NaN,NaN,36.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28.30,9.350,NaN,NaN,NaN,NaN,NaN,NaN,26.40,33.05,80.05,NaN,...,9.2,NaN,103.5,NaN,2.725,0.1,0.500000,NaN,NaN,NaN,NaN,151.5,NaN,NaN,NaN,1.666667,NaN,3.500000,1.200,4.500000,NaN,NaN,NaN,NaN,NaN,265.5,91.5,4.300000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,134.5,0.40,NaN,7.45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.018
4,141196,NaN,NaN,18.0,NaN,15.0,NaN,NaN,NaN,NaN,NaN,16.0,NaN,5.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30.0,NaN,NaN,NaN,NaN,31.70,10.300,NaN,NaN,3.0,NaN,NaN,NaN,28.10,32.50,86.40,NaN,...,9.0,NaN,97.0,NaN,0.800,NaN,0.000000,NaN,NaN,NaN,NaN,144.0,NaN,NaN,NaN,0.800000,NaN,3.000000,NaN,2.000000,NaN,7.43000,45.0,70.000,NaN,453.0,95.0,4.100000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,135.0,0.30,NaN,7.40,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
138463,3353235,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1608.0,16.0,NaN,NaN,48.5,1.55,3.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22.0,NaN,NaN,NaN,49.00,16.300,NaN,66.0,NaN,NaN,NaN,NaN,30.00,

In [36]:
# Aggregate lab features to the hospitalization level

# Get list of lab feature columns (exclude ID column)
lab_cols = lab_expanded.columns.difference(['patientunitstayid'])

# Store feature names for later use
feature_sets['labs'] = lab_cols

# Merge lab features onto patient table to recover hospitalization IDs
# Then aggregate from ICU stay → hospitalization
lab_expanded_agg = (
    patient[['patientunitstayid', 'patienthealthsystemstayid']]
    .merge(lab_expanded, on='patientunitstayid', how='left')
    .groupby('patienthealthsystemstayid')[lab_cols]
    .mean()   # average across ICU stays within the same hospitalization
    .reset_index()
)
lab_expanded_agg

,patienthealthsystemstayid,24 h urine protein,24 h urine urea nitrogen,ALT (SGPT),ANF/ANA,AST (SGOT),Acetaminophen,Amikacin peak,Amikacin random,Amikacin trough,BNP,BUN,Base Deficit,Base Excess,CPK,CPKMB,CPKMB INDEX,CRP,CRPhs,Carbamazepine,Carboxyhemoglobin,Clostridium difficile toxin A+B,Cyclosporin,Device,Digoxin,ESR,Fe,Fe/TIBC Ratio,Ferritin,FiO2,Gentamicin peak,Gentamicin random,Gentamicin trough,HCO3,HDL,HIV 1&2 AB,HSV 1&2 IgG AB,HSV 1&2 IgG AB titer,Hct,Hgb,LDH,LDL,LPM O2,Legionella pneumophila Ab,Lidocaine,Lithium,MCH,MCHC,MCV,MPV,...,calcium,cd 4,chloride,cortisol,creatinine,direct bilirubin,eos,ethanol,fibrinogen,folate,free T4,glucose,glucose CSF,haptoglobin,ionized calcium,lactate,lipase,lymphs,magnesium,monos,myoglobin,pH,paCO2,paO2,phosphate,platelets x 1000,polys,potassium,prealbumin,prolactin,protein CSF,protein C,protein S,reticulocyte count,salicylate,serum ketones,serum osmolality,sodium,total bilirubin,total cholesterol,total protein,transferrin,triglycerides,troponin I,troponin T,uric acid,urinary creatinine,urinary osmolality,urinary sodium,urinary specific gravity
0,128919,NaN,NaN,199.0,NaN,468.5,NaN,NaN,NaN,NaN,NaN,26.500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.250000,13.150,NaN,NaN,NaN,NaN,NaN,NaN,29.20,32.650000,89.350000,NaN,...,9.000000,NaN,101.500000,NaN,2.125000,NaN,0.500000,NaN,NaN,NaN,NaN,113.0,NaN,NaN,NaN,NaN,NaN,12.500000,NaN,16.500000,NaN,NaN,NaN,NaN,NaN,211.000000,70.5,4.100000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,139.000000,3.35,NaN,7.10,NaN,NaN,NaN,NaN,8.1,NaN,NaN,NaN,NaN
1,128927,NaN,NaN,52.0,NaN,40.0,NaN,NaN,NaN,NaN,NaN,15.750000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,43.600000,15.500,NaN,NaN,NaN,NaN,NaN,NaN,33.70,35.600000,94.800000,NaN,...,8.100000,NaN,107.750000,NaN,0.700000,NaN,3.000000,234.0,NaN,NaN,NaN,74.5,NaN,NaN,NaN,NaN,NaN,45.000000,1.900,7.000000,NaN,NaN,NaN,NaN,NaN,273.000000,45.0,3.750000,NaN,NaN,NaN,NaN,NaN,NaN,2.3,NaN,NaN,144.500000,0.40,NaN,7.40,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,128941,NaN,NaN,19.5,NaN,19.5,NaN,NaN,NaN,NaN,NaN,36.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28.300000,9.350,NaN,NaN,NaN,NaN,NaN,NaN,26.40,33.050000,80.050000,NaN,...,9.200000,NaN,103.500000,NaN,2.725000,0.1,0.500000,NaN,NaN,NaN,NaN,151.5,NaN,NaN,NaN,1.666667,NaN,3.500000,1.200,4.500000,NaN,NaN,NaN,NaN,NaN,265.500000,91.5,4.300000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,134.500000,0.40,NaN,7.45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.018
3,128943,NaN,NaN,18.5,NaN,16.0,NaN,NaN,NaN,NaN,24.0,15.000000,NaN,5.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30.0,NaN,NaN,NaN,NaN,34.050000,10.900,NaN,NaN,3.0,NaN,NaN,NaN,27.65,32.050000,86.250000,NaN,...,9.000000,NaN,98.000000,NaN,0.865000,NaN,0.000000,NaN,NaN,NaN,NaN,153.0,NaN,NaN,NaN,0.800000,NaN,3.000000,NaN,4.500000,NaN,7.43000,45.0,70.000,NaN,521.000000,92.5,4.100000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,137.000000,0.30,NaN,7.80,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,128948,NaN,NaN,18.0,NaN,31.5,NaN,NaN,NaN,NaN,NaN,9.333333,NaN,2.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,60.500,NaN,NaN,NaN,25.0,NaN,NaN,NaN,NaN,37.866667,11.800,NaN,NaN,NaN,NaN,NaN,NaN,28.10,31.133333,90.166667,NaN,...,8.366667,NaN,104.666667,NaN,0.426667,NaN,0.000000,NaN,NaN,4.1,NaN,130.0,NaN,NaN,NaN,3.500000,NaN,16.333333,1.550,5.666667,NaN,7.45000,37.0,51.000,NaN,473.666667,78.0,3.833333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,143.333333,0.40,NaN,6.00,NaN,NaN,0.020000,NaN,NaN,NaN,NaN,NaN,1.026
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...

In [37]:
# Merge lab features into the main aggregated dataset
patient_agg = patient_agg.merge(
    lab_expanded_agg,
    on='patienthealthsystemstayid',
    how='left'
)

# Final dataset now includes lab features
patient_agg

,patienthealthsystemstayid,gender,age,ethnicity,hospitalid,hospitaldischargeyear,mortality_at48h,hospital_numbedscategory,hospital_region,"admissiondx_ARDS-adult respiratory distress syndrome, non-cardiogenic pulmonary edema",admissiondx_Abdomen only trauma,admissiondx_Abdomen/extremity trauma,admissiondx_Abdomen/face trauma,admissiondx_Abdomen/multiple trauma,admissiondx_Abdomen/pelvis trauma,admissiondx_Abdomen/spinal trauma,admissiondx_Ablation or mapping of cardiac conduction pathway,"admissiondx_Abscess, neurologic","admissiondx_Abscess/infection-cranial, surgery for",admissiondx_Acid-base/electrolyte disturbance,admissiondx_Addisons disease,admissiondx_Adrenal neoplasm (including pheochromocytoma),admissiondx_Adrenalectomy,admissiondx_Alcohol withdrawal,admissiondx_Amputation (non-traumatic),admissiondx_Amyotrophic lateral sclerosis,admissiondx_Anaphylaxis,"admissiondx_Anastomosis, vascular",admissiondx_Anemia,"admissiondx_Aneurysm repair, ventricular","admissiondx_Aneurysm, abdominal aortic","admissiondx_Aneurysm, abdominal aortic; with dissection","admissiondx_Aneurysm, abdominal aortic; with rupture","admissiondx_Aneurysm, dissecting aortic","admissiondx_Aneurysm, thoracic aortic","admissiondx_Aneurysm, thoracic aortic; with dissection","admissiondx_Aneurysm, thoracic aortic; with rupture","admissiondx_Aneurysm/pseudoaneurysm, other","admissiondx_Aneurysms, repair of other (except ventricular)","admissiondx_Angina, stable (asymp or stable pattern of symptoms w/meds)","admissiondx_Angina, unstable (angina interferes w/quality of life or meds are tolerated poorly)",admissiondx_Aortic and Mitral valve replacement,admissiondx_Aortic valve replacement (isolated),"admissiondx_Apnea, sleep","admissiondx_Apnea-sleep; surgery for (i.e., UPPP - uvulopalatopharyngoplasty)",admissiondx_Appendectomy,"admissiondx_Arrest, respiratory (without cardiac arrest)","admissiondx_Arteriovenous malformation, surgery for","admissiondx_Arthritis, rheumatoid","admissiondx_Arthritis, septic",...,calcium,cd 4,chloride,cortisol,creatinine,direct bilirubin,eos,ethanol,fibrinogen,folate,free T4,glucose,glucose CSF,haptoglobin,ionized calcium,lactate,lipase,lymphs,magnesium,monos,myoglobin,pH,paCO2,paO2,phosphate,platelets x 1000,polys,potassium,prealbumin,prolactin,protein CSF,protein C,protein S,reticulocyte count,salicylate,serum ketones,serum osmolality,sodium,total bilirubin,total cholesterol,total protein,transferrin,triglycerides,troponin I,troponin T,uric acid,urinary creatinine,urinary osmolality,urinary sodium,urinary specific gravity
0,128919,Female,70,Caucasian,59,2015,1,<100,Midwest,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,9.000000,NaN,101.500000,NaN,2.125000,NaN,0.500000,NaN,NaN,NaN,NaN,113.0,NaN,NaN,NaN,NaN,NaN,12.500000,NaN,16.500000,NaN,NaN,NaN,NaN,NaN,211.000000,70.5,4.100000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,139.000000,3.35,NaN,7.10,NaN,NaN,NaN,NaN,8.1,NaN,NaN,NaN,NaN
1,128927,Female,52,Caucasian,60,2015,0,<100,Midwest,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,8.100000,NaN,107.750000,NaN,0.700000,NaN,3.000000,234.0,NaN,NaN,NaN,74.5,NaN,NaN,NaN,NaN,NaN,45.000000,1.900,7.000000,NaN,NaN,NaN,NaN,NaN,273.000000,45.0,3.750000,NaN,NaN,NaN,NaN,NaN,NaN,2.3,NaN,NaN,144.500000,0.40,NaN,7.40,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,128941,Male,68,Caucasian,73,2015,0,>= 500,Midwest,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,9.200000,NaN,103.500000,NaN,2.725000,0.1,0.500000,NaN,NaN,NaN,NaN,151.5,NaN,NaN,NaN,1.666667,NaN,3.500000,1.200,4.500000,NaN,NaN,NaN,NaN,NaN,265.500000,91.5,4.300000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,134.500000,0.40,NaN,7.45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.018
3,128943,Male,71,Caucasian,67,2015,0,None,Midwest,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,9.000000,NaN,98.000000,NaN,0.865000,NaN,0.000000,NaN,NaN,NaN,NaN,153.0,NaN,NaN,NaN,0.800000,NaN,3.00

In [38]:
os.getcwd()

'/Users/kara/Downloads/cs197/notebooks'

### Save your data for future usage 

In [39]:
import pickle as pk 
patient_agg.to_parquet('../data/clean_dataset.parquet', compression='gzip') # 'gzip' compression is efficient
pk.dump(feature_sets, open(f'../data/feature_names.pkl', 'wb'))

---


# 3. Section Starter: Now it's your turn! 
We have our data in `patient_agg`. Next, we'll walk through possible other tables we could have used, and explore what the data in patient_agg looks like. 

We'll end with a simple ML prediction task. 

### 1. Exploring the medication table 

In [32]:
# TODO: open the medication table and list the most common medications ordered
# BONUS: can you list the most common medications ordered in the first 24 hours of hospitalization ?
# HINT: drugorderoffset + hospitaladmitoffset (which you can get from joining on the patient table) = minutes into the hospital stay the medication was ordered 

### 2. Analyze demographic features of patient_agg

In [33]:
## TODO: Using the patient_agg table, explore and visualize the counts of the patient features, 
#            such as age, gender, ethnicity, height and weight 

### 3. Analyze hospital information 

In [34]:
## TODO 1 : Explore how many patients are recorded for each hospitalid, hospital region.  
# Hint: patient_agg already has the 'hospitalid' and 'hospital_region' column. 
# TODO 2: Why might hospital region or hospital id be an important variable to know in ML models? 

### 4. Analyzing missingness 
As we've alluded to, real-world data is imperfect. For some patients, there are features we may have no information for. b

In [35]:
## TODO 1: Examine which lab variables have the most missing data in patient_agg. 
# Hint: feature_sets['labs'] lists all the lab features in patient_agg
# Hint: .isna() is useful in pandas to see which values are missing (NaN)

In [36]:
## TODO 2: Suppose you want to use labe features in an ML model. How might you handle the missing values? 

### 5. Picking an outcome variable Y to predict 
We might care about predicting several different outcome variables. So far, we've collected a few outcome variables:
- all the diagnoses ICD10 codes made AFTER 24 hours
- mortality at 48 hours 

In [37]:
patient_agg[feature_sets['icd10_after24h']].mean().sort_values(ascending=False)[:20]

label_J96.00     0.240840
label_I10        0.181944
label_N17.9      0.168776
label_J18.9      0.137677
label_I48.0      0.125572
label_A41.9      0.096067
label_I50.9      0.095384
label_R41.82     0.087737
label_R73.9      0.080242
label_I95.9      0.078705
label_J44.9      0.071837
label_J96.91     0.067909
label_R65.21     0.067852
label_K92.2      0.065423
label_N18.9      0.054304
label_E87.2      0.053280
label_D72.829    0.051914
label_E78.5      0.049580
label_I46.9      0.046354
label_R56.9      0.046032
dtype: float64

In [38]:
Y_label = 'mortality_at48h' # TODO: change if you want ! 
# TODO: Select an outcome Y as 
# either 'mortality_at48h' or pick an ICD10 code of your choice 'label_...' (using https://www.icd10data.com/ICD10CM/Codes) 


### 5. A simple ML prediction task
Next, we will walk through a simple prediciton task using an outcome Y of your choice, using a basic logistic regression model. 

In [40]:
covariate_names = ['age', 
          'heartrate_bin1', 'heartrate_bin2', # Heart rate binned over the first 24 hours
          'respiration_bin1', 'respiration_bin2', # Respiratory rate binned over the first 24 hours
          'admissiondx_Infarction, acute myocardial (MI)', # Admitted to the ICU with MI 
          'admissiondx_Sepsis, pulmonary', # Admitted to the ICU with sepsis 
          'admissiondx_CVA, cerebrovascular accident/stroke', # Admitted to the ICU with stroke 
          'admissiondx_CHF, congestive heart failure',# Admitted to the ICU with CHF 
          ] # You can change this to be anything <24h in patient_agg! 
X = patient_agg[covariate_names].copy()
Y = patient_agg[Y_label].copy().fillna(0) # If you picked an ICD10 code as your Y, then we assume that missing values mean the patient didn't have that ICD10 code observed 
X

,age,heartrate_bin1,heartrate_bin2,respiration_bin1,respiration_bin2,"admissiondx_Infarction, acute myocardial (MI)","admissiondx_Sepsis, pulmonary","admissiondx_CVA, cerebrovascular accident/stroke","admissiondx_CHF, congestive heart failure"
0,70,132.776860,118.562500,NaN,NaN,0,0,0,0
1,52,94.763889,89.992857,NaN,NaN,0,0,0,0
2,68,92.618321,NaN,26.824427,NaN,0,0,0,0
3,71,82.141026,84.245763,21.243590,27.508621,0,1,0,0
4,77,108.300000,NaN,NaN,NaN,0,0,0,0
...,...,...,...,...,...,...,...,...,...
135773,50,89.992958,81.200000,29.207921,NaN,0,0,0,1
135774,79,90.888112,91.245283,25.454545,28.396226,0,0,0,0
135775,73,77.070922,68.592233,27.390071,24.864078,0,0,0,0
135776,81,76.178571,NaN,22.226190,NaN,0,0,0,0


In [41]:
# This distinction of numerical (continuous), binary, and categorical columns may be useful for you 
num_cols = feature_sets['labs'].tolist() + feature_sets['vitals'].tolist() + ['age', 'admissionheight', 'admissionweight']
bin_cols = feature_sets['icd10_before24h'].tolist() + feature_sets['admissiondx'].tolist()
cat_cols = ['hospital_region', 'ethnicity', 'gender', 'hospital_numbedscategory', 'hospitaldischargeyear', 'hospitalid']

# For privacy I think, the dataset sets the age of everyone over 89 as the string '>89'. We set back to a continuous number 
if 'age' in X.columns: 
    X.loc[X.age == '> 89', 'age'] = 90
    X['age'] = X['age'].astype(float)


In [42]:
### You'll find you cant just pass in X,Y into an ML model because of the missing values and potential categorical variables 
# TODO: Figure out how to handle the missing values and categorical variables (i.e., if you use `ethnicity` as a covariate in X)
# Hint: Sklearn has a few useful functions: from sklearn.preprocessing import OneHotEncoder
# or from sklearn.impute import SimpleImputer. We can choose median or mean imputation for now, 
# but we will explore in Project 2 why this might actually be bad sometimes !! 

In [43]:
### TODO: Use sklearn to build a simple logistic regression model to predict Y given X, using train_test_split to 
## separate into training and test sets. 
## Output the AUROC and AUPRC score on the test set. 
## Hint: I recommend using StandardScaler for normalizing continuous features 

In [44]:
## TODO: How well does your model do? Why might it be doing poorly / well? 

---

🎉 You did it! Congrats! 